# LeetCode 4114 — Count Consistent Subarrays II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4114 (Weekly Contest 528 — câu 4).

Bài này kết hợp hai mảng kiến thức rất hay gặp trong các bài Hard:

- **Phép toán bit AND** và quan hệ "chứa đủ các bit" (supermask / submask).
- **Kỹ thuật đếm đóng góp (contribution technique)** bằng **ngăn xếp đơn điệu (monotonic stack)**: thay vì duyệt từng đoạn con, ta hỏi *"mỗi phần tử là 'chủ' của bao nhiêu đoạn con?"*.

Ta cũng học thêm một mẹo đẹp: **với mỗi điểm cuối, AND của các đoạn con chỉ nhận rất ít giá trị khác nhau** (không quá số bit).

---

## Sau notebook này, bạn sẽ nắm được

1. Phép AND trên bit, vì sao `AND(đoạn) <= min(đoạn)` và khi nào dấu bằng xảy ra.
2. Brute Force `O(n²)` với AND và min tích lũy.
3. Cách `O(n log V)`: gom các điểm đầu theo giá trị AND.
4. Cách `O(n)`: mỗi phần tử làm "giá trị nhỏ nhất bên phải nhất" của bao nhiêu đoạn con — tìm biên trái/phải bằng ngăn xếp đơn điệu.
5. Cách xử lý **giá trị trùng nhau** để không đếm một đoạn hai lần.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng số nguyên không âm `nums`.

Một đoạn con (subarray, các phần tử **liên tiếp**) được gọi là **nhất quán (consistent)** nếu:

```text
AND của mọi phần tử trong đoạn  ==  giá trị nhỏ nhất của đoạn
```

(`AND` là phép **và theo bit**, trong Python viết là `&`.)

Đếm số đoạn con nhất quán.

---

## Ví dụ 1

```python
nums = [5, 3, 1, 2]   ->   6
```

| Đoạn | AND | min | Nhất quán? |
|---|---:|---:|:---:|
| `[5]` | 5 | 5 | ✔ |
| `[3]` | 3 | 3 | ✔ |
| `[1]` | 1 | 1 | ✔ |
| `[2]` | 2 | 2 | ✔ |
| `[3, 1]` | 1 | 1 | ✔ |
| `[5, 3, 1]` | 1 | 1 | ✔ |
| `[5, 3]` | 1 | 3 | |
| `[1, 2]` | 0 | 1 | |
| ... | | | |

Có đúng 6 đoạn nhất quán.

## Ví dụ 2

```python
nums = [1, 1, 1]   ->   6
```

Mọi đoạn đều có AND = 1 = min → cả 6 đoạn đều nhất quán.

## Ví dụ 3

```python
nums = [0]   ->   1
```

---

## Ràng buộc

```text
1 <= nums.length <= 2 * 10^5
0 <= nums[i] <= 10^6
```

Với `n = 2·10^5`, số đoạn con khoảng `2·10^10` — không thể kiểm tra từng đoạn. Ta cần `O(n)` hoặc `O(n log V)`.

# 2. Kiến thức nền tảng

## 2.1. Biểu diễn nhị phân và phép AND

Mỗi số nguyên không âm có biểu diễn nhị phân. `bin(x)` trả về chuỗi nhị phân, `format(x, '04b')` in đủ 4 chữ số.

Phép **AND theo bit** `a & b`: bit kết quả bằng 1 **khi và chỉ khi** bit tương ứng của **cả hai** số đều bằng 1.

In [1]:
for a, b in [(5, 3), (3, 1), (12, 6), (7, 5)]:
    print(f"{a:>2} = {a:04b}")
    print(f"{b:>2} = {b:04b}")
    print(f"&    {a & b:04b} = {a & b}")
    print("-" * 14)

 5 = 0101
 3 = 0011
&    0001 = 1
--------------
 3 = 0011
 1 = 0001
&    0001 = 1
--------------
12 = 1100
 6 = 0110
&    0100 = 4
--------------
 7 = 0111
 5 = 0101
&    0101 = 5
--------------


## 2.2. AND chỉ có thể **tắt** bit, không bao giờ bật thêm

Khi AND thêm một số vào, mỗi bit hoặc giữ nguyên, hoặc từ 1 thành 0. Do đó:

- `a & b <= a` và `a & b <= b`.
- AND của một đoạn **không lớn hơn bất kỳ phần tử nào** trong đoạn, nên:

```text
AND(đoạn) <= min(đoạn)        (luôn đúng)
```

Kéo dài đoạn ra thì AND chỉ có thể **giảm hoặc giữ nguyên**.

In [2]:
arr = [13, 7, 15, 5, 4]
acc = arr[0]
print(f"bắt đầu: {acc:04b} = {acc}")
for x in arr[1:]:
    acc &= x
    print(f"& {x:>2} -> {acc:04b} = {acc}")

bắt đầu: 1101 = 13
&  7 -> 0101 = 5
& 15 -> 0101 = 5
&  5 -> 0101 = 5
&  4 -> 0100 = 4


## 2.3. "Chứa đủ các bit": supermask

Ta nói `y` **chứa đủ các bit** của `x` (hay `y` là **supermask** của `x`, còn `x` là **submask** của `y`) nếu mọi bit 1 của `x` cũng là bit 1 của `y`. Cách kiểm tra:

```python
(y & x) == x
```

Ví dụ: `7 = 111` chứa đủ bit của `5 = 101` → `7 & 5 == 5`. Nhưng `3 = 011` không chứa bit cao nhất của `5`.

In [3]:
def is_supermask(y: int, x: int) -> bool:
    return (y & x) == x


print(is_supermask(7, 5))    # True  : 111 chứa 101
print(is_supermask(3, 5))    # False : 011 thiếu bit 100
print(is_supermask(5, 5))    # True  : một số luôn chứa chính nó
print(is_supermask(9, 0))    # True  : mọi số đều chứa 0 (không có bit nào cần chứa)

True
False
True
True


## 2.4. Khi nào `AND(đoạn) == min(đoạn)`?

Gọi `m = min(đoạn)`. Ta có:

> `AND(đoạn) == m`  ⇔  **mọi phần tử của đoạn đều chứa đủ các bit của `m`**.

- (⇐) Nếu mọi phần tử chứa đủ bit của `m`, thì AND vẫn giữ nguyên các bit của `m`; còn các bit khác bị tắt bởi chính phần tử `m`. Vậy AND = `m`.
- (⇒) Nếu AND = `m` thì AND mọi phần tử phải giữ đủ các bit của `m`, tức là mỗi phần tử đều có các bit đó.

Đây là chìa khóa của cả bài: điều kiện "AND bằng min" thành điều kiện **so sánh từng phần tử với một số cố định** — dễ xử lý hơn nhiều.

In [4]:
seg = [5, 3, 1]
m = min(seg)
print("min =", m, "| mọi phần tử chứa bit của min?", all(is_supermask(v, m) for v in seg))
seg = [5, 3]
m = min(seg)
print("min =", m, "| mọi phần tử chứa bit của min?", all(is_supermask(v, m) for v in seg))

min = 1 | mọi phần tử chứa bit của min? True
min = 3 | mọi phần tử chứa bit của min? False


# 3. Cách 1 — Brute Force: cố định điểm đầu, tích lũy AND và min

Với mỗi điểm đầu `i`, kéo điểm cuối `j` sang phải, cập nhật `AND` và `min` trong `O(1)` mỗi bước (không cần tính lại từ đầu).

In [5]:
from typing import List


def count_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        acc_and = nums[i]
        acc_min = nums[i]
        for j in range(i, n):
            acc_and &= nums[j]
            acc_min = min(acc_min, nums[j])
            if acc_and == acc_min:
                count += 1
    return count


print(count_bruteforce([5, 3, 1, 2]))   # 6
print(count_bruteforce([1, 1, 1]))      # 6
print(count_bruteforce([0]))            # 1

6
6
1


`O(n²)` với `n = 2·10^5` là `4·10^10` bước — quá chậm. Nhưng đây là "chuẩn đối chiếu" để kiểm tra các cách nhanh hơn.

---

# 4. Cách 2 — Gom điểm đầu theo giá trị AND: O(n log V)

## Quan sát: với một điểm cuối cố định, AND chỉ có ít giá trị khác nhau

Cố định điểm cuối `r`, cho điểm đầu `l` lùi dần từ `r` về `0`. Theo mục 2.2, `AND(nums[l..r])` **chỉ tắt thêm bit**. Mỗi lần giá trị thay đổi thì mất ít nhất một bit. Số `<= 10^6 < 2^20` có tối đa 20 bit, nên khi `l` chạy hết, AND chỉ đổi giá trị **tối đa khoảng 20 lần**.

→ Các điểm đầu `l` chia thành **không quá ~21 nhóm liên tiếp**, mỗi nhóm có cùng một giá trị AND.

## Trong một nhóm, khi nào đoạn nhất quán?

Giả sử nhóm có giá trị AND là `a`, gồm các điểm đầu `l ∈ [lo, hi]`. Vì `AND <= min` luôn đúng, `AND == min` nghĩa là **đoạn chứa một phần tử bằng đúng `a`** (khi đó min không thể lớn hơn `a`, mà cũng không nhỏ hơn AND).

Gọi `p` = vị trí xuất hiện **gần nhất** (lớn nhất, `<= r`) của giá trị `a`. Đoạn `[l, r]` chứa giá trị `a` ⇔ `l <= p`. Vậy số điểm đầu hợp lệ trong nhóm là:

```text
max(0, min(hi, p) - lo + 1)
```

## Cập nhật nhóm khi tăng `r`

Khi thêm phần tử `x = nums[r]`: mỗi nhóm cũ `(a, lo, hi)` trở thành `(a & x, lo, hi)`, rồi thêm nhóm mới `(x, r, r)`. Các nhóm kề nhau có cùng giá trị thì gộp lại.

In [6]:
def count_groups(nums: List[int]) -> int:
    last = {}            # last[v] = vị trí gần nhất của giá trị v
    groups = []          # các nhóm điểm đầu: [giá trị AND, lo, hi], lo tăng dần
    answer = 0
    for r, x in enumerate(nums):
        last[x] = r
        new_groups = []
        for a, lo, hi in groups + [[x, r, r]]:
            a &= x
            if new_groups and new_groups[-1][0] == a:
                new_groups[-1][2] = hi            # gộp vào nhóm kề có cùng AND
            else:
                new_groups.append([a, lo, hi])
        groups = new_groups
        for a, lo, hi in groups:
            p = last.get(a, -1)                   # vị trí gần nhất của giá trị a
            if p >= lo:
                answer += min(hi, p) - lo + 1
    return answer


print(count_groups([5, 3, 1, 2]))   # 6
print(count_groups([1, 1, 1]))      # 6
print(count_groups([0]))            # 1

6
6
1


## Mô phỏng với `nums = [5, 3, 1, 2]`

| r | x | các nhóm `(AND, lo, hi)` | đóng góp từng nhóm | tổng |
|---:|---:|---|---|---:|
| 0 | 5 | `(5, 0, 0)` | `5` ở vị trí 0 ≥ 0 → 1 | 1 |
| 1 | 3 | `(1, 0, 0)`, `(3, 1, 1)` | giá trị 1 chưa xuất hiện → 0; `3` ở vị trí 1 → 1 | 2 |
| 2 | 1 | `(1, 0, 2)` (ba nhóm gộp làm một) | `1` ở vị trí 2 → `2 - 0 + 1 = 3` | 5 |
| 3 | 2 | `(0, 0, 2)`, `(2, 3, 3)` | giá trị 0 chưa xuất hiện → 0; `2` ở vị trí 3 → 1 | 6 |

Ở `r = 2`: các đoạn `[5,3,1]`, `[3,1]`, `[1]` đều có AND = 1 và chứa số 1 → cả 3 đều nhất quán.

In [7]:
def trace_groups(nums: List[int]) -> int:
    last, groups, answer = {}, [], 0
    for r, x in enumerate(nums):
        last[x] = r
        new_groups = []
        for a, lo, hi in groups + [[x, r, r]]:
            a &= x
            if new_groups and new_groups[-1][0] == a:
                new_groups[-1][2] = hi
            else:
                new_groups.append([a, lo, hi])
        groups = new_groups
        parts = []
        for a, lo, hi in groups:
            p = last.get(a, -1)
            add = min(hi, p) - lo + 1 if p >= lo else 0
            answer += add
            parts.append(f"(AND={a}, l∈[{lo},{hi}]) +{add}")
        print(f"r = {r}, x = {x}: " + ", ".join(parts) + f" | tổng = {answer}")
    return answer


trace_groups([5, 3, 1, 2]);

r = 0, x = 5: (AND=5, l∈[0,0]) +1 | tổng = 1
r = 1, x = 3: (AND=1, l∈[0,0]) +0, (AND=3, l∈[1,1]) +1 | tổng = 2
r = 2, x = 1: (AND=1, l∈[0,2]) +3 | tổng = 5
r = 3, x = 2: (AND=0, l∈[0,2]) +0, (AND=2, l∈[3,3]) +1 | tổng = 6


Mỗi bước xử lý tối đa ~21 nhóm → `O(n log V)` với `V = max(nums)`. Đã đủ nhanh, nhưng còn một cách `O(n)` rất đẹp.

---

# 5. Bước đột phá: đếm theo "chủ" của mỗi đoạn

Thay vì duyệt đoạn con, hãy gán **mỗi đoạn nhất quán cho đúng một phần tử** rồi đếm mỗi phần tử "sở hữu" bao nhiêu đoạn.

Chọn "chủ" của đoạn là **vị trí giá trị nhỏ nhất, lấy vị trí bên phải nhất nếu có nhiều vị trí cùng bằng min**. Mỗi đoạn có đúng một chủ như vậy → không đếm trùng.

Xét phần tử `i` với giá trị `x = nums[i]`. Đoạn `[l, r]` (chứa `i`) **nhất quán và có chủ là `i`** khi và chỉ khi:

- **Bên trái** (`l <= k < i`): mọi `nums[k]` **chứa đủ bit của `x`** (được phép bằng `x`, vì `i` vẫn là vị trí bên phải nhất).
- **Bên phải** (`i < k <= r`): mọi `nums[k]` **chứa đủ bit của `x` và khác `x`** (nếu bằng `x` thì vị trí đó mới là chủ).

(Chứa đủ bit của `x` thì `>= x`, nên `x` đúng là min; khác `x` thì `> x`.)

Gọi:

- `left[i]` = vị trí gần nhất bên trái **vi phạm** điều kiện trái (hoặc `-1`),
- `right[i]` = vị trí gần nhất bên phải **vi phạm** điều kiện phải (hoặc `n`).

Khi đó `l` chọn được trong `(left[i], i]` và `r` trong `[i, right[i])`:

```text
đóng góp của i = (i - left[i]) * (right[i] - i)
```

Đáp án = tổng đóng góp.

## Kiểm chứng bằng ví dụ 1 `[5, 3, 1, 2]`

| i | x | nhị phân | left[i] | right[i] | đóng góp | các đoạn |
|---:|---:|---|---:|---:|---:|---|
| 0 | 5 | 101 | -1 | 1 | `1 · 1 = 1` | `[5]` |
| 1 | 3 | 011 | 0 | 2 | `1 · 1 = 1` | `[3]` |
| 2 | 1 | 001 | -1 | 3 | `3 · 1 = 3` | `[1]`, `[3,1]`, `[5,3,1]` |
| 3 | 2 | 010 | 2 | 4 | `1 · 1 = 1` | `[2]` |

Tổng `6` ✔.

# 6. Tìm `left` và `right` bằng ngăn xếp đơn điệu

Câu hỏi "**vị trí gần nhất bên trái không thỏa điều kiện**" là dạng kinh điển của **ngăn xếp đơn điệu**:

Duyệt từ trái sang phải, giữ một stack các chỉ số "ứng viên". Khi tới `i` với giá trị `x`:

1. **Pop** mọi chỉ số trên đỉnh có giá trị **chứa đủ bit của `x`** (`(top & x) == x`) — chúng không phải là "vi phạm" đối với `x`.
2. Đỉnh còn lại (nếu có) là `left[i]`; rỗng thì `left[i] = -1`.
3. Push `i`.

**Vì sao được phép pop?** Nếu phần tử `p` bị pop vì chứa đủ bit của `x`, thì sau này một giá trị `y` nào đó:

- nếu `x` chứa đủ bit của `y` thì `p` cũng chứa đủ bit của `y` (tính bắc cầu) → `p` cũng không phải đáp án của `y`;
- nếu `x` không chứa đủ bit của `y` thì `i` (gần hơn `p`) đã là vi phạm → ta dừng ở `i`, không cần tới `p`.

Vậy `p` không bao giờ cần nữa. Mỗi chỉ số được push và pop **tối đa một lần** → `O(n)`.

Với `right`, duyệt từ phải sang trái, pop khi đỉnh **chứa đủ bit của `x` và khác `x`**. Quan hệ "chứa đủ bit và khác nhau" cũng có tính bắc cầu nên lập luận y hệt.

In [8]:
def boundaries(nums: List[int]):
    n = len(nums)
    left = [-1] * n
    st = []
    for i, x in enumerate(nums):
        while st and (nums[st[-1]] & x) == x:          # đỉnh chứa đủ bit của x -> bỏ
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)

    right = [n] * n
    st = []
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while st and nums[st[-1]] != x and (nums[st[-1]] & x) == x:
            st.pop()
        right[i] = st[-1] if st else n
        st.append(i)
    return left, right


left, right = boundaries([5, 3, 1, 2])
print("left  =", left)
print("right =", right)

left  = [-1, 0, -1, 2]
right = [1, 2, 3, 4]


# 7. Cách 3 — Đếm đóng góp: O(n)

In [9]:
def count_stack(nums: List[int]) -> int:
    left, right = boundaries(nums)
    return sum((i - left[i]) * (right[i] - i) for i in range(len(nums)))


print(count_stack([5, 3, 1, 2]))   # 6
print(count_stack([1, 1, 1]))      # 6
print(count_stack([0]))            # 1

6
6
1


## Mô phỏng với giá trị trùng: `nums = [12, 4, 6, 4, 5]`

Nhị phân: `12 = 1100`, `4 = 0100`, `6 = 0110`, `5 = 0101` — mọi số đều chứa bit `0100` của số 4.

| i | x | left[i] | right[i] | đóng góp | giải thích |
|---:|---:|---:|---:|---:|---|
| 0 | 12 | -1 | 1 | 1 | bên phải là 4, không chứa bit `1000` |
| 1 | 4 | -1 | 3 | `2 · 2 = 4` | bên phải dừng ở số 4 tiếp theo (vị trí 3) |
| 2 | 6 | 1 | 3 | 1 | hai bên đều là 4, không chứa bit `0010` |
| 3 | 4 | -1 | 5 | `4 · 2 = 8` | bên trái được đi qua cả số 4 ở vị trí 1 |
| 4 | 5 | 3 | 5 | 1 | |

Tổng `15`. Để ý số 4 ở vị trí 1 "nhường" các đoạn chứa cả hai số 4 cho số 4 ở vị trí 3 (bên phải nhất) — nhờ vậy không đoạn nào bị đếm hai lần.

In [10]:
def trace_stack(nums: List[int]) -> int:
    left, right = boundaries(nums)
    total = 0
    for i, x in enumerate(nums):
        c = (i - left[i]) * (right[i] - i)
        total += c
        print(f"i = {i}, x = {x:>2} ({x:04b}): l ∈ ({left[i]}, {i}], r ∈ [{i}, {right[i]}) "
              f"-> ({i} - ({left[i]})) * ({right[i]} - {i}) = {c}")
    print("Tổng:", total, "| brute force:", count_bruteforce(nums))
    return total


trace_stack([12, 4, 6, 4, 5]);

i = 0, x = 12 (1100): l ∈ (-1, 0], r ∈ [0, 1) -> (0 - (-1)) * (1 - 0) = 1
i = 1, x =  4 (0100): l ∈ (-1, 1], r ∈ [1, 3) -> (1 - (-1)) * (3 - 1) = 4
i = 2, x =  6 (0110): l ∈ (1, 2], r ∈ [2, 3) -> (2 - (1)) * (3 - 2) = 1
i = 3, x =  4 (0100): l ∈ (-1, 3], r ∈ [3, 5) -> (3 - (-1)) * (5 - 3) = 8
i = 4, x =  5 (0101): l ∈ (3, 4], r ∈ [4, 5) -> (4 - (3)) * (5 - 4) = 1
Tổng: 15 | brute force: 15


# 8. Phiên bản đúng format LeetCode

In [11]:
from typing import List


class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        n = len(nums)
        left = [-1] * n
        st = []
        for i, x in enumerate(nums):
            while st and (nums[st[-1]] & x) == x:
                st.pop()
            left[i] = st[-1] if st else -1
            st.append(i)

        answer = 0
        st = []
        for i in range(n - 1, -1, -1):
            x = nums[i]
            while st and nums[st[-1]] != x and (nums[st[-1]] & x) == x:
                st.pop()
            right = st[-1] if st else n
            st.append(i)
            answer += (i - left[i]) * (right - i)
        return answer

In [12]:
sol = Solution()

tests = [
    ([5, 3, 1, 2], 6, "ví dụ 1"),
    ([1, 1, 1], 6, "ví dụ 2"),
    ([0], 1, "ví dụ 3"),
    ([0, 0, 0, 0], 10, "toàn số 0: mọi đoạn đều nhất quán"),
    ([7, 3, 1], 6, "7 ⊇ 3 ⊇ 1: mọi đoạn nhất quán"),
    ([1, 2, 4, 8], 4, "các bit rời nhau: chỉ đoạn độ dài 1"),
    ([12, 4, 6, 4, 5], 15, "giá trị trùng"),
]
for nums, expected, name in tests:
    result = sol.countConsistent(nums)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:38} -> {result}")

PASS | ví dụ 1                                -> 6
PASS | ví dụ 2                                -> 6
PASS | ví dụ 3                                -> 1
PASS | toàn số 0: mọi đoạn đều nhất quán      -> 10
PASS | 7 ⊇ 3 ⊇ 1: mọi đoạn nhất quán          -> 6
PASS | các bit rời nhau: chỉ đoạn độ dài 1    -> 4
PASS | giá trị trùng                          -> 15


In [13]:
import random
import time

random.seed(4114)
big = [random.randint(0, 10**6) for _ in range(2 * 10**5)]
t0 = time.perf_counter()
r3 = sol.countConsistent(big)
t3 = time.perf_counter() - t0
t0 = time.perf_counter()
r2 = count_groups(big)
t2 = time.perf_counter() - t0
print(f"n = 2·10^5: stack = {r3} ({t3:.2f}s) | gom nhóm AND = {r2} ({t2:.2f}s)")

n = 2·10^5: stack = 201179 (0.05s) | gom nhóm AND = 201179 (0.20s)


# 9. Vì sao đúng?

1. **Điều kiện tương đương** (mục 2.4): đoạn nhất quán ⇔ mọi phần tử chứa đủ bit của min.
2. **Mỗi đoạn có đúng một chủ**: vị trí bên phải nhất đạt min. Do đó tổng đóng góp đếm mỗi đoạn nhất quán đúng một lần.
3. **Đếm đúng các đoạn của chủ `i`**: một đoạn `[l, r]` chứa `i` có chủ là `i` và nhất quán ⇔ mọi phần tử bên trái `i` (trong đoạn) chứa đủ bit của `x`, mọi phần tử bên phải chứa đủ bit của `x` và khác `x`. Hai điều kiện độc lập theo `l` và theo `r`, và mỗi điều kiện đúng trên một khoảng liên tục kề `i` → số đoạn là tích độ dài hai khoảng.
4. **Stack tìm đúng biên** (mục 6) nhờ tính bắc cầu của quan hệ "chứa đủ bit".

---

## Trực giác

> Mỗi phần tử `x` tự hỏi: *"Mình có thể làm 'ông chủ nhỏ nhất' của một đoạn kéo dài tới đâu?"* — sang trái được chừng nào hàng xóm còn chứa đủ bit của mình, sang phải được chừng nào hàng xóm còn chứa đủ bit **và** không phải bản sao của mình.

# 10. Độ phức tạp

Gọi `n = len(nums)`, `V = max(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | AND, min tích lũy theo điểm đầu | `O(n²)` | `O(1)` |
| 2. Gom nhóm AND | ≤ log V nhóm điểm đầu cho mỗi điểm cuối | `O(n log V)` | `O(n + log V)` |
| 3. Đóng góp + stack | biên trái/phải bằng ngăn xếp đơn điệu | `O(n)` | `O(n)` |

# 11. Đối chiếu ba cách trên dữ liệu ngẫu nhiên

In [14]:
random.seed(2026)
for _ in range(3000):
    nums = [random.randint(0, 15) for _ in range(random.randint(1, 12))]
    expected = count_bruteforce(nums)
    assert count_groups(nums) == expected, nums
    assert count_stack(nums) == expected, nums
    assert sol.countConsistent(nums) == expected, nums
print("3000 bộ test ngẫu nhiên: 4 hàm cho cùng kết quả!")

3000 bộ test ngẫu nhiên: 4 hàm cho cùng kết quả!


# 12. Những lỗi dễ mắc

## Lỗi 1 — Không phân biệt "bằng `x`" ở hai phía → đếm trùng

Nếu bên phải cũng cho phép phần tử bằng `x`, các đoạn chứa hai số bằng nhau sẽ được đếm cho **cả hai** vị trí.

In [15]:
def wrong_double_count(nums):
    n = len(nums)
    left, _ = boundaries(nums)
    right = [n] * n
    st = []
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while st and (nums[st[-1]] & x) == x:        # SAI: thiếu nums[top] != x
            st.pop()
        right[i] = st[-1] if st else n
        st.append(i)
    return sum((i - left[i]) * (right[i] - i) for i in range(n))


print("Sai :", wrong_double_count([1, 1, 1]))
print("Đúng:", sol.countConsistent([1, 1, 1]))

Sai : 10
Đúng: 6


## Lỗi 2 — Dùng điều kiện "lớn hơn hoặc bằng" thay cho "chứa đủ bit"

Ngăn xếp tìm "phần tử nhỏ hơn gần nhất" (như bài *Sum of Subarray Minimums*) là đúng cho min, nhưng **không đủ** cho bài này: `6 >= 5` nhưng `6 = 110` không chứa bit `001` của `5`, nên `AND(5, 6) = 4 != 5`.

In [16]:
def wrong_only_compare(nums):
    n = len(nums)
    left = [-1] * n
    st = []
    for i, x in enumerate(nums):
        while st and nums[st[-1]] >= x:              # SAI: chỉ so sánh lớn/nhỏ
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    right = [n] * n
    st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] > nums[i]:
            st.pop()
        right[i] = st[-1] if st else n
        st.append(i)
    return sum((i - left[i]) * (right[i] - i) for i in range(n))


print("Sai :", wrong_only_compare([5, 6]))       # đếm cả [5, 6] (AND = 4 != 5)
print("Đúng:", sol.countConsistent([5, 6]))

Sai : 3
Đúng: 2


## Lỗi 3 — Viết `nums[st[-1]] & x == x` mà nghĩ là `(nums[st[-1]] & x) == x`

Trong Python, `&` có độ ưu tiên **cao hơn** `==`, nên biểu thức không ngoặc vẫn đúng. Nhưng trong C++/Java thì `==` được tính trước `&` → sai hoàn toàn. Thói quen tốt: **luôn đặt ngoặc** quanh phép bit.

## Lỗi 4 — Quên rằng số 0 là "chứa được" bởi mọi số

`(y & 0) == 0` luôn đúng, nên một số 0 có thể làm chủ đoạn kéo dài tới tận các số 0 khác. Test `[0, 0, 0, 0] -> 10` bắt lỗi này nếu ai đó xử lý 0 như trường hợp đặc biệt sai.

## Lỗi 5 — Brute Force `O(n²)` với `n = 2·10^5`

Đúng nhưng TLE.

# 13. Mẹo nhận diện

1. Đề yêu cầu **đếm đoạn con** với điều kiện liên quan **min/max** → nghĩ tới **đếm đóng góp**: mỗi phần tử làm min (max) của bao nhiêu đoạn → tìm biên trái/phải bằng **ngăn xếp đơn điệu**.
2. Khi có giá trị trùng, dùng quy ước **"bên phải nhất"** (một phía `<`, phía kia `<=`) để mỗi đoạn có đúng một chủ.
3. Đề có **AND / OR** của đoạn con → nhớ hai tính chất:
   - AND chỉ tắt bit, OR chỉ bật bit khi kéo dài đoạn;
   - với mỗi điểm cuối, AND (hoặc OR) của các đoạn kết thúc tại đó chỉ có **≤ số bit** giá trị khác nhau.
4. Điều kiện "AND bằng min" ⇔ "mọi phần tử chứa đủ bit của min" — luôn thử **biến đổi điều kiện tổng thể thành điều kiện trên từng phần tử**.

Các bài cùng họ: *Sum of Subarray Minimums* (LeetCode 907), *Number of Subarrays With AND Value of K* (3209), *Bitwise ORs of Subarrays* (898), và phiên bản dễ hơn *Count Consistent Subarrays I* (4111).

# 14. Biến thể

## Biến thể 1 — "OR của đoạn bằng max của đoạn"

Đối xứng hoàn toàn: OR ≥ max luôn đúng, và OR = max ⇔ mọi phần tử là **submask** của max (`(x | y) == x`, tức `(y & ~x) == 0`). Chủ của đoạn là vị trí max bên phải nhất.

In [17]:
def count_or_equals_max(nums: List[int]) -> int:
    n = len(nums)
    is_sub = lambda y, x: (y | x) == x               # y là submask của x
    left = [-1] * n
    st = []
    for i, x in enumerate(nums):
        while st and is_sub(nums[st[-1]], x):
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    total = 0
    st = []
    for i in range(n - 1, -1, -1):
        x = nums[i]
        while st and nums[st[-1]] != x and is_sub(nums[st[-1]], x):
            st.pop()
        right = st[-1] if st else n
        st.append(i)
        total += (i - left[i]) * (right - i)
    return total


def brute_or_max(nums):
    n, c = len(nums), 0
    for i in range(n):
        o, m = 0, -1
        for j in range(i, n):
            o |= nums[j]
            m = max(m, nums[j])
            c += o == m
    return c


random.seed(1)
for _ in range(1000):
    arr = [random.randint(0, 15) for _ in range(random.randint(1, 10))]
    assert count_or_equals_max(arr) == brute_or_max(arr)
print(count_or_equals_max([1, 3, 2, 7]), "| khớp brute force trên 1000 test")

10 | khớp brute force trên 1000 test


## Biến thể 2 — Tổng của min trên mọi đoạn nhất quán

Chỉ cần nhân đóng góp với giá trị: `Σ x_i · (i - left[i]) · (right[i] - i)`.

In [18]:
def sum_min_consistent(nums: List[int]) -> int:
    left, right = boundaries(nums)
    return sum(nums[i] * (i - left[i]) * (right[i] - i) for i in range(len(nums)))


print(sum_min_consistent([5, 3, 1, 2]))   # 5 + 3 + 1·3 + 2 = 13

13


# 15. Template tư duy

```python
# Đếm đóng góp: mỗi i là "chủ" của bao nhiêu đoạn
left = [-1] * n; st = []
for i, x in enumerate(nums):
    while st and <nums[st[-1]] "không chặn" x>:      # phía trái: cho phép bằng
        st.pop()
    left[i] = st[-1] if st else -1
    st.append(i)
right = [n] * n; st = []
for i in reversed(range(n)):
    while st and <nums[st[-1]] "không chặn" x, và khác x>:   # phía phải: nghiêm ngặt
        st.pop()
    right[i] = st[-1] if st else n
    st.append(i)
answer = sum((i - left[i]) * (right[i] - i) for i in range(n))
```

Điều kiện "không chặn" phải có **tính bắc cầu** thì stack mới đúng.

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [6, 4, 4, 7]` (`110, 100, 100, 111`), lập bảng `i | x | left | right | đóng góp`. Đáp án là bao nhiêu? Kiểm tra bằng cell dưới.

In [19]:
# Kiểm tra Bài 1
trace_stack([6, 4, 4, 7]);

i = 0, x =  6 (0110): l ∈ (-1, 0], r ∈ [0, 1) -> (0 - (-1)) * (1 - 0) = 1
i = 1, x =  4 (0100): l ∈ (-1, 1], r ∈ [1, 2) -> (1 - (-1)) * (2 - 1) = 2
i = 2, x =  4 (0100): l ∈ (-1, 2], r ∈ [2, 4) -> (2 - (-1)) * (4 - 2) = 6
i = 3, x =  7 (0111): l ∈ (2, 3], r ∈ [3, 4) -> (3 - (2)) * (4 - 3) = 1
Tổng: 10 | brute force: 10


## Bài 2 — Dự đoán output

```python
print(12 & 10, 12 | 10)
print((13 & 5) == 5, (13 & 6) == 6)
acc = 15
for v in [14, 7, 13]:
    acc &= v
    print(acc, end=" ")
```

In [20]:
print(12 & 10, 12 | 10)
print((13 & 5) == 5, (13 & 6) == 6)
acc = 15
for v in [14, 7, 13]:
    acc &= v
    print(acc, end=" ")

8 14
True False
14 6 4 

## Bài 3 — Hoàn thiện hàm tìm biên trái

In [21]:
def exercise_3_left_bound(nums: List[int]) -> List[int]:
    left = [-1] * len(nums)
    st = []
    for i, x in enumerate(nums):
        # TODO 1: pop khi đỉnh chứa đủ bit của x
        # TODO 2: left[i] = đỉnh hoặc -1
        # TODO 3: push i
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_left_bound([5, 3, 1, 2]) == [-1, 0, -1, 2]

## Bài 4 — Đoạn nhất quán dài nhất

Trả về độ dài lớn nhất của một đoạn nhất quán.

```python
longest_consistent([5, 3, 1, 2])      -> 3     # [5, 3, 1]
longest_consistent([12, 4, 6, 4, 5])  -> 5     # cả mảng
```

In [22]:
def exercise_4_longest_consistent(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_longest_consistent([5, 3, 1, 2]) == 3
# assert exercise_4_longest_consistent([12, 4, 6, 4, 5]) == 5

## Bài 5 — Đếm đoạn có `AND == k` cho trước

Đếm số đoạn con có AND **đúng bằng** `k` (bài *Number of Subarrays With AND Value of K*, LeetCode 3209). Dùng ý tưởng gom nhóm ở cách 2.

```python
count_and_equals_k([1, 1, 1], 1)   -> 6
count_and_equals_k([1, 1, 2], 1)   -> 3
count_and_equals_k([1, 2, 3], 2)   -> 2
```

In [23]:
def exercise_5_count_and_equals_k(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_and_equals_k([1, 1, 1], 1) == 6
# assert exercise_5_count_and_equals_k([1, 1, 2], 1) == 3
# assert exercise_5_count_and_equals_k([1, 2, 3], 2) == 2

## Bài 6 — Mini challenge: tổng số phần tử của mọi đoạn nhất quán

Tính tổng **độ dài** của tất cả các đoạn nhất quán, trong `O(n)`.

```python
total_length([5, 3, 1, 2])   -> 1+1+1+1+2+3 = 9
```

Gợi ý: với chủ `i`, các đoạn là `[l, r]` với `l ∈ (L, i]`, `r ∈ [i, R)`. Tổng `(r - l + 1)` trên mọi cặp có công thức đóng.

In [24]:
def exercise_6_total_length(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_total_length([5, 3, 1, 2]) == 9

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: điều kiện pop là `(nums[st[-1]] & x) == x`.
- Mức 2: sau vòng `while`, `left[i] = st[-1] if st else -1`.
- Mức 3: `st.append(i)` và cuối hàm `return left`.

**Bài 4**

- Mức 1: đoạn dài nhất có chủ `i` là `[left[i] + 1, right[i] - 1]`.
- Mức 2: độ dài `right[i] - left[i] - 1`.
- Mức 3: lấy `max` trên mọi `i`.

**Bài 5**

- Mức 1: duy trì các nhóm `(AND, số điểm đầu)` cho mỗi điểm cuối.
- Mức 2: mỗi bước: AND từng nhóm với `x`, thêm nhóm `(x, 1)`, gộp nhóm trùng giá trị.
- Mức 3: cộng số điểm đầu của nhóm có giá trị `k`.

**Bài 6**

- Mức 1: gọi `A = i - L` (số cách chọn `l`), `B = R - i` (số cách chọn `r`).
- Mức 2: `Σ (r - l + 1) = Σ_l Σ_r ((r - i) + (i - l) + 1)`.
- Mức 3: `= A·(0+1+...+(B-1)) + B·(0+1+...+(A-1)) + A·B`.

# 17. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

`[6, 4, 4, 7]` = `110, 100, 100, 111`:

| i | x | left | right | đóng góp |
|---:|---:|---:|---:|---:|
| 0 | 6 | -1 | 1 | `1 · 1 = 1` |
| 1 | 4 | -1 | 2 | `2 · 1 = 2` (dừng ở số 4 tiếp theo) |
| 2 | 4 | -1 | 4 | `3 · 2 = 6` |
| 3 | 7 | 2 | 4 | `1 · 1 = 1` |

Tổng `10`.

## Đáp án Bài 2

```text
8 14
True False
14 6 4
```

In [25]:
# Đáp án Bài 3
def answer_3_left_bound(nums: List[int]) -> List[int]:
    left = [-1] * len(nums)
    st = []
    for i, x in enumerate(nums):
        while st and (nums[st[-1]] & x) == x:
            st.pop()
        left[i] = st[-1] if st else -1
        st.append(i)
    return left


assert answer_3_left_bound([5, 3, 1, 2]) == [-1, 0, -1, 2]
print("Bài 3: PASS")

Bài 3: PASS


In [26]:
# Đáp án Bài 4
def answer_4_longest_consistent(nums: List[int]) -> int:
    left, right = boundaries(nums)
    return max(right[i] - left[i] - 1 for i in range(len(nums)))


assert answer_4_longest_consistent([5, 3, 1, 2]) == 3
assert answer_4_longest_consistent([12, 4, 6, 4, 5]) == 5
print("Bài 4: PASS")

Bài 4: PASS


In [27]:
# Đáp án Bài 5
def answer_5_count_and_equals_k(nums: List[int], k: int) -> int:
    groups = []                  # [giá trị AND, số điểm đầu]
    total = 0
    for x in nums:
        new_groups = []
        for a, c in groups + [[x, 1]]:
            a &= x
            if new_groups and new_groups[-1][0] == a:
                new_groups[-1][1] += c
            else:
                new_groups.append([a, c])
        groups = new_groups
        total += sum(c for a, c in groups if a == k)
    return total


def brute_and_k(nums, k):
    n, c = len(nums), 0
    for i in range(n):
        a = -1
        for j in range(i, n):
            a = nums[j] if a == -1 else a & nums[j]
            c += a == k
    return c


assert answer_5_count_and_equals_k([1, 1, 1], 1) == 6
assert answer_5_count_and_equals_k([1, 1, 2], 1) == 3
assert answer_5_count_and_equals_k([1, 2, 3], 2) == 2
random.seed(5)
for _ in range(500):
    arr = [random.randint(0, 7) for _ in range(random.randint(1, 10))]
    kk = random.randint(0, 7)
    assert answer_5_count_and_equals_k(arr, kk) == brute_and_k(arr, kk)
print("Bài 5: PASS (kể cả 500 test ngẫu nhiên)")

Bài 5: PASS (kể cả 500 test ngẫu nhiên)


In [28]:
# Đáp án Bài 6
def answer_6_total_length(nums: List[int]) -> int:
    left, right = boundaries(nums)
    total = 0
    for i in range(len(nums)):
        A = i - left[i]
        B = right[i] - i
        total += A * (B * (B - 1) // 2) + B * (A * (A - 1) // 2) + A * B
    return total


def brute_total_length(nums):
    n, s = len(nums), 0
    for i in range(n):
        a, m = nums[i], nums[i]
        for j in range(i, n):
            a &= nums[j]
            m = min(m, nums[j])
            if a == m:
                s += j - i + 1
    return s


assert answer_6_total_length([5, 3, 1, 2]) == 9
random.seed(6)
for _ in range(500):
    arr = [random.randint(0, 15) for _ in range(random.randint(1, 10))]
    assert answer_6_total_length(arr) == brute_total_length(arr)
print("Bài 6: PASS (kể cả 500 test ngẫu nhiên)")

Bài 6: PASS (kể cả 500 test ngẫu nhiên)


# 18. Tổng kết

Bài **Count Consistent Subarrays II** dạy ba ý quan trọng:

1. **Biến đổi điều kiện**: `AND == min` ⇔ mọi phần tử chứa đủ bit của min.
2. **Đếm đóng góp**: gán mỗi đoạn cho chủ là vị trí min bên phải nhất; đóng góp = `(i - left) · (right - i)`.
3. **Ngăn xếp đơn điệu** với một quan hệ có tính bắc cầu ("chứa đủ bit") để tìm biên trong `O(n)`.

Và một công cụ dự phòng: với mỗi điểm cuối, AND của các đoạn chỉ có `≤ log V` giá trị.

## Công thức nhớ nhanh

```text
trái : pop khi (top & x) == x
phải : pop khi top != x và (top & x) == x
answer = Σ (i - left[i]) · (right[i] - i)
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(n)
```

# 19. Checklist trước khi submit

- [ ] Điều kiện pop dùng **"chứa đủ bit"** `(top & x) == x`, không phải so sánh lớn/nhỏ?
- [ ] Một phía cho phép **bằng `x`**, phía kia **không** (tránh đếm trùng)?
- [ ] Biên mặc định: `left = -1`, `right = n`?
- [ ] Đặt ngoặc quanh phép bit khi so sánh?
- [ ] Đã test: toàn số 0, toàn số bằng nhau, các bit rời nhau, giá trị trùng?
- [ ] Không dùng `O(n²)` cho `n = 2·10^5`?